# RESILINK GRID v4 — TinyML Clinical Risk Model Training & Validation
**Author:** Yashwanth (Lead: Embedded, Crypto, Radio, Edge AI)  
**Target Hardware:** NXP FRDM-MCXN236 (Arm Cortex-M33 @ 150MHz)  
**Principle:** AI predicts, rules decide, logs prove.  
**Notice:** Decision support, not a diagnosis.

In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score
from data_generator import generate_physiological_cohort
from news2_engine import calculate_news2

print('Libraries successfully imported.')

## 1. Physiological Dataset Synthesis
Generates multi-scenario 1 Hz physiological time series (Normal, Tachycardia, Hypoxia, Sepsis, Shock Collapse) based on AR(1) physiological drift models.

In [2]:
df = generate_physiological_cohort(num_samples_per_scenario=3000, seed=42)
print(f'Total synthetic records: {len(df)}')
df.head()

## 2. Feature Preprocessing & Standardization
12 physiological input features: instant values + 10-second rolling means and slopes.

In [3]:
FEATURE_NAMES = [
    'hr', 'spo2', 'temp', 'resp',
    'hr_mean10', 'spo2_mean10', 'temp_mean10', 'resp_mean10',
    'hr_slope10', 'spo2_slope10', 'temp_slope10', 'resp_slope10'
]
X = df[FEATURE_NAMES].values
y = df['target_class'].values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
print(f'Train shape: {X_train_scaled.shape}, Test shape: {X_test_scaled.shape}')

## 3. Neural Network Architecture & Training
Compact 2-layer MLP (12 -> 16 -> 8 -> 4) constrained for Cortex-M33 execution within 16 KB tensor arena.

In [4]:
model = MLPClassifier(
    hidden_layer_sizes=(16, 8),
    activation='relu',
    solver='adam',
    max_iter=400,
    random_state=42,
    learning_rate_init=0.005
)
model.fit(X_train_scaled, y_train)
print('Model training complete.')

## 4. Evaluation: Accuracy, Sensitivity & Confusion Matrix

In [5]:
CLASS_NAMES = ['LOW', 'MED', 'HIGH', 'CRITICAL']
y_pred = model.predict(X_test_scaled)
print(f'Accuracy: {accuracy_score(y_test, y_pred) * 100:.2f}%')
print(classification_report(y_test, y_pred, target_names=CLASS_NAMES))

cm = confusion_matrix(y_test, y_pred)
# Safety metric for Critical (P0)
tp = cm[3, 3]
fn = sum(cm[3, :]) - tp
sensitivity = tp / (tp + fn)
print(f'Critical (P0) Sensitivity: {sensitivity * 100:.2f}%')

## 5. INT8 Symmetric Quantization & C Header Generation
Converts float32 weights and biases into INT8 / INT32 fixed-point tables for embedded microcontrollers.

In [6]:
from train_risk_model import write_risk_c_header

quantized_weights = []
weight_scales = []
for idx, (W, b) in enumerate(zip(model.coefs_, model.intercepts_)):
    max_val = max(np.max(np.abs(W)), np.max(np.abs(b)))
    scale = max_val / 127.0 if max_val > 0 else 1.0
    W_int8 = np.clip(np.round(W / scale), -128, 127).astype(np.int8)
    b_int32 = np.round(b / scale).astype(np.int32)
    quantized_weights.append((W_int8, b_int32))
    weight_scales.append(scale)

write_risk_c_header('risk_model_data.h', scaler.mean_, scaler.scale_, quantized_weights, weight_scales)
print('C header generated: risk_model_data.h')